# 03 — Model and sanity checks

## Setup

In [36]:
import os, sys, pathlib, warnings
root = pathlib.Path.cwd()
if root.name == "notebooks":
    root = root.parent
    os.chdir(root)
sys.path.insert(0, str(root))

# Warnings print the full path of the file that raised them.
warnings.filterwarnings("ignore", module="tqdm")
warnings.filterwarnings("ignore", message=".*IProgress.*")

## Parameters

In [37]:
import torch, yaml
from src.model import UNet, build_model, ARCHITECTURES

cfg = yaml.safe_load(open("configs/default.yaml"))
model = build_model(cfg["model"])
print(f"{cfg['model']['arch']}: {sum(p.numel() for p in model.parameters()):,} parameters")
print("available:", ARCHITECTURES)

unet_scratch: 7,763,041 parameters
available: ('unet_scratch', 'unet_resnet34', 'deeplabv3plus_resnet34')


## Architecture

In [38]:
m = UNet(in_channels=3, base_channels=32, depth=4)
print("encoder   :", [b.block[0].out_channels for b in m.encoders])
print("bottleneck:", m.bottleneck.block[0].out_channels)
print("decoder   :", [b.block[0].out_channels for b in m.decoders])
print("head      :", m.head.out_channels)

encoder   : [32, 64, 128, 256]
bottleneck: 512
decoder   : [256, 128, 64, 32]
head      : 1


### Shapes

In [39]:
x = torch.randn(2, 3, 256, 256)
y = m(x)
print("in  ", tuple(x.shape))
print("out ", tuple(y.shape))
print("range", round(float(y.min()), 2), "to", round(float(y.max()), 2))

in   (2, 3, 256, 256)
out  (2, 1, 256, 256)
range -2.05 to 1.19


### Input size must divide by 16

In [40]:
try:
    m(torch.randn(1, 3, 250, 250))
except ValueError as e:
    print("ValueError:", e)

ValueError: Input size (250, 250) must be divisible by 16


## Overfit one batch

Settings: `notebook.overfit` in the config.

In [41]:
from src.losses import BCEDiceLoss
from src.metrics import BinaryMetrics

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

images = torch.randn(4, 3, 128, 128)
masks = torch.zeros(4, 1, 128, 128)
for i in range(4):
    t, l = 20 + 10 * i, 30 + 8 * i
    masks[i, :, t:t + 50, l:l + 40] = 1.0
    images[i, :, t:t + 50, l:l + 40] += 3.0
images, masks = images.to(device), masks.to(device)

print("device:", device)
print("batch :", tuple(images.shape))
print(f"buildings: {masks.mean():.1%} of pixels")

device: cuda
batch : (4, 3, 128, 128)
buildings: 12.2% of pixels


In [42]:
nb = cfg["notebook"]["overfit"]
print(nb)

net = UNet(base_channels=nb["base_channels"], depth=nb["depth"]).to(device)
opt = torch.optim.AdamW(net.parameters(), lr=nb["learning_rate"])
loss_fn = BCEDiceLoss()

net.train()
for step in range(nb["steps"]):
    opt.zero_grad(set_to_none=True)
    logits = net(images)
    loss = loss_fn(logits, masks)
    loss.backward()
    opt.step()
    if step % 50 == 0:
        mt = BinaryMetrics(); mt.update(logits.detach(), masks)
        print(f"step {step:4d}  loss {loss.item():.4f}  IoU {mt.compute()['iou']:.4f}")

{'base_channels': 16, 'depth': 3, 'learning_rate': 0.003, 'steps': 201}
step    0  loss 1.7655  IoU 0.1098
step   50  loss 1.0230  IoU 0.9442
step  100  loss 0.7450  IoU 0.9999
step  150  loss 0.4634  IoU 1.0000
step  200  loss 0.2625  IoU 1.0000


In [43]:
mt = BinaryMetrics()
with torch.no_grad():
    mt.update(net(images), masks)
final = mt.compute()
print({k: round(v, 4) for k, v in final.items()})
print("PASS" if final["iou"] > 0.95 else "FAIL")

{'iou': 1.0, 'dice': 1.0, 'precision': 1.0, 'recall': 1.0}
PASS


## Loss: BCE + Dice

In [44]:
from torch.nn import functional as F
from src.losses import dice_loss

truth = torch.zeros(1, 1, 100, 100); truth[:, :, 20:50, 20:50] = 1.0
blank = torch.full((1, 1, 100, 100), -5.0)

print("prediction: all background")
print("  BCE ", round(F.binary_cross_entropy_with_logits(blank, truth).item(), 4))
print("  Dice", round(dice_loss(blank, truth).item(), 4))

prediction: all background
  BCE  0.4567
  Dice 0.9865
